# Paper 2 — Lexical Suppression: Filler Validity Control

**Runtime: GPU, hours. Resumable.**

The validity control for the *lexical* suppression mechanism: is suppression-necessity actually higher at revisions than at matched filler positions, or does banning those tokens degrade generation everywhere?

Fillers are matched 1:1 on model and marker word, and GPT-4o-verified as non-revisions. This is the lexical counterpart to notebook 4's steering control — running both is what lets the two mechanisms be compared on the same events.

**Prerequisites:** `labels/`, `rejects/`, `traces/`, and the pilot event sets.

> **Scoring caveat.** This notebook scores rollouts with the parser it was written against. The paper's corrected figures come from notebook 6, which re-scores the saved rollout text with the fixed parser and repaired ground truth. Treat this notebook's own printed p-values as provisional.


In [ ]:
# ---- Google Drive -----------------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")

## Filler control


In [ ]:
# =============================================================================
# FILLER CONTROL FOR NECESSITY — all three stages in one file
#
# WHY THIS CONTROL EXISTS
# Necessity is measured by intervention, not validated against an external
# criterion. Without a control we cannot exclude the alternative explanation:
# "suppression degrades generation generally, and measured necessity is
# perturbation damage rather than anything specific to revisions."
#
# This runs the identical necessity pipeline at MATCHED filler positions --
# same model, same marker word, but GPT-4o-verified NOT genuine revisions --
# mirroring Task 1's MATCHED contrast logic.
#
#   revision necessity >> filler necessity -> revision-specific measurement.
#   revision necessity == filler necessity -> generic degradation; the
#       "most revisions are not load-bearing" claim must then be withdrawn
#       or heavily qualified.
#
# -----------------------------------------------------------------------------
# HOW TO RUN — set STAGES below, then run the cell/file.
#
#   STAGES = [1]        Select matched filler events.   CPU, seconds.
#                       ** RUN THIS FIRST AND READ THE OUTPUT. ** Check the
#                       under-matched cells and per-model composition before
#                       committing GPU hours to stage 2.
#   STAGES = [2]        Generate suppressed rollouts.   GPU, hours. Resumable:
#                       safe to interrupt and re-run, it skips completed events.
#   STAGES = [3]        Compare the two arms.           CPU, instant.
#   STAGES = [1,2,3]    Everything, in order.
#
# Heavy imports (torch/transformers) are loaded only when stage 2 runs, so
# stages 1 and 3 work on a CPU runtime without them.
# =============================================================================

STAGES = [1]          # <-- edit this



from pathlib import Path
from collections import defaultdict, Counter
import json, random, re, time

import numpy as np
from scipy.stats import mannwhitneyu

In [ ]:
# ---------------------------------------------------------------- paths ----
GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
TRACES_DIR  = GDRIVE_BASE / "traces"
REJECT_DIR  = GDRIVE_BASE / "rejects"
EXPORT_DIR  = GDRIVE_BASE / "hand_labeling_export"

REVISION_EVENTS_PATH  = EXPORT_DIR / "counterfactual_pilot_events.json"
REVISION_RESULTS_PATH = EXPORT_DIR / "counterfactual_rollout_results.json"

In [ ]:
FILLER_EVENTS_PATH    = EXPORT_DIR / "filler_control_pilot_events.json"
FILLER_RESULTS_PATH   = EXPORT_DIR / "filler_control_rollout_results.json"
COMPARISON_OUT_PATH   = EXPORT_DIR / "revision_vs_filler_necessity.json"

# ------------------------------------------------------------ parameters ----
SELECT_SEED            = 5150   # dedicated to this control; not reused
N_ROLLOUTS             = 8      # must match the revision arm
ROLLOUT_MAX_NEW_TOKENS = 2048   # must match the revision arm

In [ ]:
SAMPLE_TEMPERATURE     = 0.6
SAMPLE_TOP_P           = 0.95

# Identical to the revision arm's ban list -- both arms must be suppressed and
# leak-filtered the same way or the comparison is confounded.
SUPPRESSED_PHRASES = [
    "wait", "Wait", "actually", "Actually", "let me reconsider",
    "that's wrong", "that is wrong", "i made an error", "I made an error",
    "let me try again", "this doesn't work", "this does not work",
    "i need to reconsider", "I need to reconsider", "hmm", "Hmm",
    "that's not right", "that is not right", "let me rethink",
    "on second thought", "let me go back", "wait, let me", "Wait, let me",
    "actually, let me", "Actually, let me", "let me recheck", "let me re-check",
    "alternatively", "Alternatively", "I made a mistake", "i made a mistake",
    "I think I made a mistake", "let me double check", "let me double-check",
    "Let me double check", "Let me double-check",
    "hold on", "Hold on",
]

# lowercase form used for post-hoc leak detection in stage 3
EXPLICITLY_BANNED = sorted({p.lower() for p in SUPPRESSED_PHRASES})


# ============================================================================
# shared helpers
# ============================================================================
def normalize_marker(text):
    """Matches Task 6 Phase 1's normalization, so 'Wait,' and 'wait' group as
    the same marker family."""
    return re.sub(r"[^\w\s]", "", str(text).lower()).strip()


def is_leaked(text):
    t = text.lower()
    return any(p in t for p in EXPLICITLY_BANNED)

In [ ]:
def extract_answer(text: str) -> str:
    """Verbatim from Task_1_Data_Pipeline.ipynb -- 'correct' must mean exactly
    what it meant when the original traces were generated."""
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        return boxed[-1].strip()
    m = re.search(r'(?:answer is|therefore)[:\s]*([0-9A-Za-z\-\.\/]+)',
                  text, re.IGNORECASE)
    return m.group(1).strip() if m else "UNKNOWN"


def build_prompt(problem: dict, tokenizer, model_name: str) -> str:
    """Verbatim from Task_1_Data_Pipeline.ipynb -- must match exactly or every
    downstream token index shifts."""
    is_reasoning = any(tag in model_name for tag in ["R1", "QwQ"])
    if problem.get("difficulty") == "hard":
        instruction = ("Solve the following competition mathematics problem carefully. "
                        "Show your reasoning step by step and verify your answer.")
    else:
        instruction = ("Solve the following mathematics competition problem. "
                        "Show your working step by step.")
    if is_reasoning:
        messages = [{"role": "user", "content": f"{instruction}\n\nProblem: {problem['problem']}"}]
    else:
        messages = [
            {"role": "system", "content": "You are an expert mathematician. Solve problems with "
                                          "clear step-by-step reasoning."},
            {"role": "user", "content": f"{instruction}\n\nProblem: {problem['problem']}"}
        ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def necessity_from_rollouts(rollout_correct):
    if not rollout_correct:
        return None
    return 1.0 - (sum(rollout_correct) / len(rollout_correct))


# ============================================================================
# STAGE 1 — select matched filler control events
# ============================================================================
def stage1_select():
    print(f"\n{'#'*70}\n# STAGE 1 — SELECT MATCHED FILLER CONTROL EVENTS\n{'#'*70}\n")
    assert REVISION_EVENTS_PATH.exists(), f"{REVISION_EVENTS_PATH} not found"
    assert REVISION_RESULTS_PATH.exists(), f"{REVISION_RESULTS_PATH} not found"

    with open(REVISION_EVENTS_PATH) as f:
        revision_events = json.load(f)
    with open(REVISION_RESULTS_PATH) as f:
        rollout_results = json.load(f)

    # only match against revision events that actually received a score
    scored = {(r["model"], r["problem_id"]) for r in rollout_results}
    revision_events = [e for e in revision_events if (e["model"], e["problem_id"]) in scored]
    print(f"Revision events with necessity scores: {len(revision_events)}")

    demand = Counter((e["model"], normalize_marker(e["marker_phrase"])) for e in revision_events)
    print(f"Distinct (model, marker) cells to match: {len(demand)}")

    print("\nBuilding filler pool...")
    filler_pool = defaultdict(list)
    n_total = n_not_correct = n_truncated = n_no_trace = 0

    for model_dir in sorted(TRACES_DIR.iterdir()):
        if not model_dir.is_dir():
            continue
        model = model_dir.name
        reject_path = REJECT_DIR / f"{model}_confirmed_filler.json"
        if not reject_path.exists():
            continue
        with open(reject_path) as f:
            fillers = json.load(f)

        trace_cache = {}
        for flr in fillers:
            n_total += 1
            pid = flr["problem_id"]
            if pid not in trace_cache:
                tp = model_dir / f"{pid}.json"
                trace_cache[pid] = json.load(open(tp)) if tp.exists() else None
            trace = trace_cache[pid]
            if trace is None:
                n_no_trace += 1;      continue
            if not trace.get("correct", False):
                n_not_correct += 1;   continue
            if trace.get("truncated", False):
                n_truncated += 1;     continue

            cs = flr.get("char_start")
            full_response = trace.get("full_response", "")
            if cs is None or not (0 <= cs < len(full_response)):
                continue

            filler_pool[(model, normalize_marker(flr["matched_text"]))].append({
                "problem_id": pid, "model": model,
                "human_label": "filler",
                "marker_phrase": flr["matched_text"],
                "char_start": cs,
                "problem": trace.get("problem", ""),
                "difficulty": trace.get("difficulty", "hard"),
                "answer_gt": trace.get("answer_gt", ""),
                "prefix_response_text": full_response[:cs],
            })

    print(f"  filler positions scanned  : {n_total}")
    print(f"  excluded, trace incorrect : {n_not_correct}")
    print(f"  excluded, trace truncated : {n_truncated}")
    print(f"  excluded, no trace file   : {n_no_trace}")
    print(f"  eligible cells            : {len(filler_pool)}")
    print(f"  eligible positions        : {sum(len(v) for v in filler_pool.values())}")

    random.seed(SELECT_SEED)
    selected, unmatched = [], []
    for (model, marker), n_needed in sorted(demand.items()):
        avail = filler_pool.get((model, marker), [])
        if not avail:
            unmatched.append((model, marker, n_needed, 0));  continue
        take = min(n_needed, len(avail))
        if take < n_needed:
            unmatched.append((model, marker, n_needed, take))
        selected.extend(random.sample(avail, take))

    print(f"\n{'='*70}\nMATCHED DRAW\n{'='*70}")
    print(f"  revision events to match : {len(revision_events)}")
    print(f"  filler controls selected : {len(selected)}")
    if unmatched:
        print(f"\n  under-matched cells ({len(unmatched)}):")
        for model, marker, needed, got in unmatched[:15]:
            print(f"    {model:<32} {marker!r:<22} needed {needed}, got {got}")
        print("  Report the matched fraction alongside the result.")

    print("\n  control composition by model:")
    for m, c in sorted(Counter(e["model"] for e in selected).items()):
        print(f"    {m:<32} {c}")
    print("\n  control composition by marker (top 10):")
    for mk, c in Counter(normalize_marker(e["marker_phrase"]) for e in selected).most_common(10):
        print(f"    {mk!r:<22} {c}")

    with open(FILLER_EVENTS_PATH, "w") as f:
        json.dump(selected, f, indent=2)
    print(f"\nWrote {len(selected)} filler control events -> {FILLER_EVENTS_PATH}")
    print("Review the composition above before running stage 2 (GPU hours).")


# ============================================================================
# STAGE 2 — generate suppressed rollouts at filler positions
# ============================================================================
def stage2_generate():
    print(f"\n{'#'*70}\n# STAGE 2 — GENERATE SUPPRESSED ROLLOUTS (filler arm)\n{'#'*70}\n")
    # heavy imports only when this stage actually runs
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    assert FILLER_EVENTS_PATH.exists(), f"{FILLER_EVENTS_PATH} not found -- run stage 1 first"
    with open(FILLER_EVENTS_PATH) as f:
        events = json.load(f)
    print(f"Loaded {len(events)} filler control events.")

    def build_bad_words_ids(tokenizer):
        out, seen = [], set()
        for phrase in SUPPRESSED_PHRASES:
            for variant in (phrase, " " + phrase):
                ids = tuple(tokenizer.encode(variant, add_special_tokens=False))
                if ids and ids not in seen:
                    seen.add(ids); out.append(list(ids))
        return out

    @torch.no_grad()
    def generate_suppressed(model, tokenizer, prefix_text, bad_words_ids, seed):
        torch.manual_seed(seed)
        inputs = tokenizer(prefix_text, return_tensors="pt", truncation=False).to(model.device)
        out = model.generate(
            **inputs, max_new_tokens=ROLLOUT_MAX_NEW_TOKENS, do_sample=True,
            temperature=SAMPLE_TEMPERATURE, top_p=SAMPLE_TOP_P,
            bad_words_ids=bad_words_ids,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
        return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)

    by_model = defaultdict(list)
    for ev in events:
        by_model[ev["model"]].append(ev)

    all_results, already_done = [], set()
    if FILLER_RESULTS_PATH.exists():
        with open(FILLER_RESULTS_PATH) as f:
            all_results = json.load(f)
        already_done = {(r["problem_id"], r["model"]) for r in all_results}
        print(f"Resuming: {len(already_done)} events already completed.")

    for model_short, model_events in by_model.items():
        remaining = [e for e in model_events if (e["problem_id"], model_short) not in already_done]
        if not remaining:
            print(f"\n{model_short}: all events done -- skipping model load.")
            continue
        model_full = f"deepseek-ai/{model_short}" if not model_short.startswith("deepseek-ai/") else model_short
        print(f"\n{'='*60}\n{model_full}  ({len(remaining)} events remaining)\n{'='*60}")

        tokenizer = AutoTokenizer.from_pretrained(model_full, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_full, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
        model.eval()

        bad_words_ids = build_bad_words_ids(tokenizer)
        print(f"Built {len(bad_words_ids)} banned token sequences. First 5 decoded:")
        for ids in bad_words_ids[:5]:
            print(f"  {ids} -> {tokenizer.decode(ids)!r}")

        for ev in remaining:
            print(f"\n  {ev['problem_id']}  (filler control)")
            prompt = build_prompt(
                {"problem": ev["problem"], "difficulty": ev["difficulty"]}, tokenizer, model_full)
            prefix = prompt + ev["prefix_response_text"]
            correct, texts = [], []
            t0 = time.time()
            for r in range(N_ROLLOUTS):
                txt = generate_suppressed(model, tokenizer, prefix, bad_words_ids, seed=1000 + r)
                ans = extract_answer(txt)
                ok = ans.strip() == ev["answer_gt"].strip()
                correct.append(ok); texts.append(txt)
                print(f"    rollout {r+1}/{N_ROLLOUTS}: ans={ans!r} (gt={ev['answer_gt']!r}) correct={ok}")
            print(f"  necessity={necessity_from_rollouts(correct):.3f}  ({time.time()-t0:.1f}s)")

            all_results.append({
                "problem_id": ev["problem_id"], "model": model_short,
                "human_label": "filler", "rollout_correct": correct, "rollout_texts": texts,
            })
            with open(FILLER_RESULTS_PATH, "w") as f:
                json.dump(all_results, f, indent=2)

        del model
        torch.cuda.empty_cache()

    print(f"\nWrote {len(all_results)} filler results -> {FILLER_RESULTS_PATH}")


# ============================================================================
# STAGE 3 — compare the two arms
# ============================================================================
def _scores(path, arm_name):
    with open(path) as f:
        results = json.load(f)
    scores, lost_events, lost_rollouts = [], 0, 0
    for r in results:
        if "rollout_texts" in r:
            clean = [c for c, t in zip(r["rollout_correct"], r["rollout_texts"]) if not is_leaked(t)]
            lost_rollouts += len(r["rollout_correct"]) - len(clean)
        else:
            clean = r["rollout_correct"]
        if not clean:
            lost_events += 1;  continue
        scores.append({"model": r["model"], "problem_id": r["problem_id"],
                       "necessity": necessity_from_rollouts(clean)})
    print(f"{arm_name}: {len(scores)} events "
          f"({lost_rollouts} leaked rollouts dropped, {lost_events} events lost)")
    return scores

### Run

Edit `STAGES` above, run that cell, then the one below. Repeat: `[0]` → `[1]` → `[2]` → `[3]`.


In [ ]:
# ---- WHICH STAGES TO RUN  (edit, run this cell, then run the next) ---------
# Overrides the STAGES value in the config cell above so you can change it
# between runs without scrolling back.
#
#   [0] preflight    [1] select + generate revision-arm rollouts (GPU, hours)
#   [2] filler-arm rollouts (GPU, hours)    [3] analysis (fast)
#
# Generation stages are resumable -- interrupt freely and re-run.

STAGES = [0]

print(f"STAGES = {STAGES}")

In [ ]:
def _describe(scores, label):
    v = np.array([s["necessity"] for s in scores])
    nz = int((v == 0).sum())
    print(f"\n  {label}  (n={len(v)})")
    print(f"    mean {v.mean():.3f}   median {np.median(v):.3f}   sd {v.std():.3f}")
    print(f"    exactly zero            : {nz}/{len(v)} ({nz/len(v):.0%})")
    print(f"    fully necessary (1.000) : {int((v==1.0).sum())}/{len(v)}")
    return v


def stage3_compare():
    print(f"\n{'#'*70}\n# STAGE 3 — REVISION vs. FILLER NECESSITY\n{'#'*70}\n")
    assert REVISION_RESULTS_PATH.exists(), f"{REVISION_RESULTS_PATH} not found"
    assert FILLER_RESULTS_PATH.exists(), f"{FILLER_RESULTS_PATH} not found -- run stage 2 first"

    rev = _scores(REVISION_RESULTS_PATH, "revision arm")
    flr = _scores(FILLER_RESULTS_PATH,   "filler arm  ")

    print(f"\n{'='*70}\nNECESSITY BY ARM\n{'='*70}")
    v_rev = _describe(rev, "REVISION positions")
    v_flr = _describe(flr, "FILLER positions (matched control)")

    if len(v_flr) < 5:
        print("\nToo few filler events for a meaningful comparison. Stopping.")
        return

    u, p = mannwhitneyu(v_rev, v_flr, alternative="two-sided")
    auc = u / (len(v_rev) * len(v_flr))

    print(f"\n{'='*70}\nARM COMPARISON\n{'='*70}")
    print(f"  Mann-Whitney U p = {p:.4f}")
    print(f"  P(random revision > random filler) = {auc:.3f}")
    print(f"  rank-biserial effect size = {2*auc-1:+.3f}")
    print(f"  median difference (revision - filler) = {np.median(v_rev)-np.median(v_flr):+.3f}")

    rng = np.random.RandomState(0)
    diffs = [np.median(rng.choice(v_rev, len(v_rev), True)) -
             np.median(rng.choice(v_flr, len(v_flr), True)) for _ in range(5000)]
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"  95% CI on median difference (bootstrap): [{lo:+.3f}, {hi:+.3f}]")

    print("\n  per-model medians:")
    bmr, bmf = defaultdict(list), defaultdict(list)
    for s in rev: bmr[s["model"]].append(s["necessity"])
    for s in flr: bmf[s["model"]].append(s["necessity"])
    for m in sorted(set(bmr) | set(bmf)):
        r_, f_ = bmr.get(m, []), bmf.get(m, [])
        rm = f"{np.median(r_):.3f}" if r_ else "  -  "
        fm = f"{np.median(f_):.3f}" if f_ else "  -  "
        print(f"    {m:<32} revision {rm} (n={len(r_):>3})   filler {fm} (n={len(f_):>3})")

    print(f"\n{'#'*70}\n# READ\n{'#'*70}")
    if p < 0.05 and np.median(v_rev) > np.median(v_flr):
        print("Revision positions show HIGHER necessity than matched filler positions.")
        print("The measurement is revision-specific -- suppression effects are not merely")
        print("generic perturbation damage. The 'most revisions are not load-bearing'")
        print("finding is properly controlled and can be reported as such.")
    elif p >= 0.05:
        print("NO detectable difference between revision and matched filler positions.")
        print("Adverse outcome: measured necessity appears to reflect generic degradation")
        print("from suppression rather than anything specific to revisions. The 'most")
        print("revisions are not load-bearing' claim must be withdrawn or heavily")
        print("qualified -- as stated it presumes a revision-specific measurement, which")
        print("this does not support. Check the CI width before calling it conclusive.")
    else:
        print("Filler positions show HIGHER necessity than revision positions -- unexpected;")
        print("understand this before interpreting either arm.")

    with open(COMPARISON_OUT_PATH, "w") as f:
        json.dump({
            "n_revision": len(v_rev), "n_filler": len(v_flr),
            "median_revision": float(np.median(v_rev)), "median_filler": float(np.median(v_flr)),
            "mean_revision": float(v_rev.mean()), "mean_filler": float(v_flr.mean()),
            "frac_zero_revision": float((v_rev == 0).mean()),
            "frac_zero_filler": float((v_flr == 0).mean()),
            "mannwhitney_p": float(p), "prob_revision_gt_filler": float(auc),
            "rank_biserial": float(2*auc-1), "median_diff_ci95": [float(lo), float(hi)],
        }, f, indent=2)
    print(f"\nWrote {COMPARISON_OUT_PATH}")


# ============================================================================
if __name__ == "__main__":
    if 1 in STAGES: stage1_select()
    if 2 in STAGES: stage2_generate()
    if 3 in STAGES: stage3_compare()
    print(f"\nDone. Stages run: {STAGES}")